<a href="https://colab.research.google.com/github/Jdk30/research_project/blob/main/DS_Pipeline_Script_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
DS_Pipeline_Script_Python.py

Colab Biomedical DS Pipeline — Training, Grad-CAM, Clustering, and Cohort Attention Masks

PROJECT CONTEXT (Down syndrome phenotyping, cohort-aware):
--------------------------------------------------------------------
Step 0 – Environment & configuration
Step 1 – Dataset discovery and cohort inference
Step 2 – Datasets, transforms, and dataloaders
Step 3 – Model definitions and feature extraction
Step 4 – Training, evaluation, and model selection
Step 5 – Grad-CAM and population attention masks
Step 6 – Image enhancement and composite grids
Step 7 – Cohort mean faces and pairwise comparisons
Step 8 – Clustering and cohort associations
Step 9 – Orchestration (run_pipeline)
Step 10 – Script entry point
--------------------------------------------------------------------
"""

# ====================================================================
# SECTION 0: SETUP, IMPORTS, AND CONFIGURATION
# ====================================================================

# --- 0.1 Dependency Installation (run in a separate Colab cell) ---
!pip install --quiet "numpy<1.27" "pandas<2.2" "protobuf<4.26" scikit-image
!pip install --quiet mediapipe opencv-python-headless --no-deps



In [ ]:
import os, sys, math, time, json, random, shutil, warnings, re, io
from pathlib import Path
from itertools import combinations

# Optional Colab Drive mount
try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        print("[INFO] Mounting Google Drive...")
        drive.mount("/content/drive")
        print("[INFO] Drive mounted successfully.")
except Exception as e:
    print(f"[WARN] Google Drive not mounted automatically ({e}). "
          "If you are running in Colab, mount it manually if needed.")

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageFilter

import matplotlib
matplotlib.use("Agg")  # headless-safe
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.datasets.folder import default_loader

from sklearn.metrics import (
    confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score,
    silhouette_score, classification_report
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Optional libraries
try:
    from skimage.color import rgb2lab, lab2rgb
    from skimage import exposure, restoration
except Exception:
    rgb2lab = lab2rgb = exposure = restoration = None
    print("[WARN] skimage utilities not fully available.")

try:
    import mediapipe as mp  # type: ignore
    if "cv2" not in sys.modules:
        import cv2  # type: ignore
    print("[INFO] MediaPipe and OpenCV available for face alignment.")
except Exception:
    mp = None
    cv2 = None
    print("[WARN] MediaPipe/OpenCV not available for face alignment.")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")



[INFO] MediaPipe and OpenCV available for face alignment.
[INFO] Using device: cuda


In [ ]:
# --- Seed & filesystem helpers ------------------------------------------------

def set_all_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def ensure_project_tree(project_dir: str) -> Path:
    root = Path(project_dir)
    (root / "data").mkdir(parents=True, exist_ok=True)
    (root / "artifacts" / "figures").mkdir(parents=True, exist_ok=True)
    (root / "artifacts" / "models").mkdir(parents=True, exist_ok=True)
    (root / "artifacts" / "composites").mkdir(parents=True, exist_ok=True)
    (root / "artifacts" / "clustering" / "overall").mkdir(parents=True, exist_ok=True)
    (root / "artifacts" / "clustering" / "pairwise").mkdir(parents=True, exist_ok=True)
    (root / "artifacts" / "mean_faces").mkdir(parents=True, exist_ok=True)
    (root / "reports").mkdir(parents=True, exist_ok=True)
    return root

def save_fig(path, tight: bool = True, dpi: int = 300) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if tight:
        plt.tight_layout()
    png_path = str(path.with_suffix(".png"))
    pdf_path = str(path.with_suffix(".pdf"))
    plt.savefig(png_path, dpi=dpi, bbox_inches="tight")
    try:
        plt.savefig(pdf_path, dpi=dpi, bbox_inches="tight")
    except Exception:
        pass
    plt.close()

def to_device(x, device: torch.device):
    if isinstance(x, (list, tuple)):
        return [to_device(t, device) for t in x]
    return x.to(device, non_blocking=True)

def disable_inplace_relu(model: nn.Module) -> None:
    for m in model.modules():
        if isinstance(m, nn.ReLU):
            m.inplace = False





In [ ]:
# --- Config -------------------------------------------------------------------

DEFAULT_CFG = {
    "project_dir": "/content/drive/MyDrive/biomed_pipeline_updated_run",
    "dataset_root": "/content/drive/MyDrive/t21_dataset1",
    "img_size": 224,
    "seed": 42,
    "quick_mode": False,

    "models": ["mobilenet_v2", "densenet121", "resnet50", "convnext_tiny", "vgg16"],

    "training": {
        "batch_size": 32,
        "num_workers": 2,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "max_epochs": 10,
        "patience": 3,
        "limit_train_batches": 1.0,
        "limit_val_batches": 1.0,
        "mixup": 0.0,
        "label_smoothing": 0.0,
        "class_weights": True,
    },

    "viz": {
        "gradcam_n": 8,
        "composite_n": 12,
    },

    "calibrate": False,
    "tta": False,

    "clustering": {
        "n_components_pca": 50,
        "k_candidates": [2, 3, 4, 5, 6],
        "pairwise_enabled": True,
        "overall_enabled": True,
    },

    "mean_face": {
        "enabled": True,
        "cohorts": ["Congo", "Rwanda", "Guadeloupe", "Public"],
        "img_size": 256,
        "max_per_cohort": 600,
        "align": "mediapipe",
        "mask_oval": True,
        "splits": ["train", "valid", "test"],
    },

    "mean_face_compare": {
        "enabled": True,
        "score": "mse",
        "max_images_per_pair": 400,
    },
}




In [ ]:
# ====================================================================
# SECTION 2: DATA DISCOVERY AND CLASS INFERENCE
# ====================================================================

_POS_PATTERNS = [
    r"(?i)(?<![a-z])downsyndrome(?![a-z])",
    r"(?i)(?<![a-z])down(?![a-z])",
    r"(?i)(?<![a-z])trisomy21(?![a-z])",
    r"(?i)(?<![a-z])t21(?![a-z])",
    r"(?i)(?<![a-z])ds(?![a-z])",
]
_NEG_PATTERNS = [
    r"(?i)(?<![a-z])nondownsyndrome(?![a-z])",
    r"(?i)(?<![a-z])non[_\-]?down(?![a-z])",
    r"(?i)(?<![a-z])nondown(?![a-z])",
    r"(?i)(?<![a-z])controls?(?![a-z])",
    r"(?i)(?<![a-z])normal(?![a-z])",
    r"(?i)(?<![a-z])healthy(?![a-z])",
    r"(?i)(?<![a-z])healty(?![a-z])",
    r"(?i)(?<![a-z])negative(?![a-z])",
    r"(?i)(?<![a-z])nd(?![a-z])",
]
_POS_RE = [re.compile(p) for p in _POS_PATTERNS]
_NEG_RE = [re.compile(p) for p in _NEG_PATTERNS]

def _infer_class_from_tokens(path_str: str):
    fname = Path(path_str).name.lower()
    has_pos = any(r.search(fname) for r in _POS_RE)
    has_neg = any(r.search(fname) for r in _NEG_RE)
    if has_pos and not has_neg:
        return "downsyndrome"
    if has_neg and not has_pos:
        return "nondownsyndrome"
    return None

COHORT_RULES = {
    "Congo":       set(list(range(1, 11)) + list(range(23, 45))),
    "Guadeloupe":  set(range(11, 23)),
    "Rwanda":      set(range(45, 74)),
}

def infer_cohort_from_path(path_str: str) -> str:
    fname = Path(path_str).name.lower()
    if "healthy" in fname or "healty" in fname:
        return "nondownsyndrome"
    if re.search(r"(?i)down_", fname) and not re.search(r"(?i)down[_-]?\d+", fname):
        return "Public"
    m = re.search(r"(?i)down[_-]?(\d+)\b", fname)
    if m:
        num = int(m.group(1))
        for cohort, nums in COHORT_RULES.items():
            if num in nums:
                return cohort
        return "Public"
    return "Public"

def discover_dataset(root: str):
    root = Path(root)
    splits = {s: root / s for s in ("train", "valid", "test")}
    for s, p in splits.items():
        if not p.exists():
            raise RuntimeError(f"Missing split folder: {p}")

    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".jfif", ".webp"}

    def has_class_subdirs(split_dir: Path) -> bool:
        return any(d.is_dir() for d in split_dir.iterdir())

    def scan_with_subdirs(split_dir: Path, split_name: str):
        rows = []
        for cls_dir in sorted([d for d in split_dir.iterdir() if d.is_dir()]):
            for imgp in cls_dir.rglob("*"):
                if imgp.suffix.lower() in IMG_EXTS:
                    rows.append({
                        "split": split_name,
                        "class_name": cls_dir.name,
                        "path": str(imgp),
                    })
        return rows

    def scan_flat_and_infer(split_dir: Path, split_name: str):
        rows = []
        unknown = 0
        samples = []
        for imgp in split_dir.rglob("*"):
            if imgp.is_dir() or imgp.suffix.lower() not in IMG_EXTS:
                continue
            cls = _infer_class_from_tokens(str(imgp))
            if cls is None:
                unknown += 1
                if len(samples) < 5:
                    samples.append(str(imgp))
                continue
            rows.append({
                "split": split_name,
                "class_name": cls,
                "path": str(imgp),
            })
        if unknown > 0:
            print(f"[WARN] {unknown} images dropped (ambiguous class tokens). "
                  f"Examples: {samples}")
        return rows

    rows = []
    for split_name, split_dir in splits.items():
        all_imgs = [
            p for p in split_dir.rglob("*")
            if (p.is_file() and p.suffix.lower() in IMG_EXTS)
        ]
        class_dirs = [d for d in split_dir.iterdir() if d.is_dir()]
        print(f"[Probe] {split_name}: dir={split_dir} | "
              f"class_dirs={len(class_dirs)} | imgs_found={len(all_imgs)}")
        if has_class_subdirs(split_dir):
            rows += scan_with_subdirs(split_dir, split_name)
        else:
            rows += scan_flat_and_infer(split_dir, split_name)

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No images found under expected structure or token rules.")

    classes = sorted(df["class_name"].unique().tolist())
    class_to_id = {c: i for i, c in enumerate(classes)}
    df["label"] = df["class_name"].map(class_to_id)

    trainval = df[df["split"].isin(["train", "valid"])].copy()
    testdf = df[df["split"] == "test"].copy()

    print("\n[Discovery Summary]")
    print(trainval.groupby(["split", "class_name"]).size())

    return trainval, testdf, class_to_id

In [ ]:
# ====================================================================
# SECTION 3: DATASETS, DATALOADERS, AND TRANSFORMS
# ====================================================================

def imagenet_train_tfms(img_size: int):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1)
        ], p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        ),
    ])

def imagenet_val_tfms(img_size: int):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        ),
    ])

class ImageCSV(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int = 224, train: bool = False):
        self.df = df.reset_index(drop=True)
        self.tfms = imagenet_train_tfms(img_size) if train else imagenet_val_tfms(img_size)
        self.loader = default_loader

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = self.loader(row["path"]).convert("RGB")
        x = self.tfms(img)
        y = int(row["label"])
        return x, y, row["path"]

def make_dataloaders(train_df: pd.DataFrame, val_df: pd.DataFrame, cfg: dict):
    bs = cfg["training"]["batch_size"]
    nw = cfg["training"]["num_workers"]
    img_size = cfg["img_size"]

    train_ds = ImageCSV(train_df, img_size=img_size, train=True)
    val_ds   = ImageCSV(val_df,   img_size=img_size, train=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=bs,
        shuffle=True,
        num_workers=nw,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=bs,
        shuffle=False,
        num_workers=nw,
        pin_memory=True,
    )
    return train_ds, val_ds, train_loader, val_loader


In [ ]:
# ====================================================================
# SECTION 4: MODEL DEFINITIONS AND FEATURE EXTRACTION
# ====================================================================

def make_model(name: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    name = name.lower()

    if name == "mobilenet_v2":
        m = torchvision.models.mobilenet_v2(
            weights=torchvision.models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        )
        m.classifier[1] = nn.Linear(m.last_channel, num_classes)
        return m

    if name == "densenet121":
        m = torchvision.models.densenet121(
            weights=torchvision.models.DenseNet121_Weights.DEFAULT if pretrained else None
        )
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        return m

    if name == "resnet50":
        m = torchvision.models.resnet50(
            weights=torchvision.models.ResNet50_Weights.DEFAULT if pretrained else None
        )
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m

    if name == "convnext_tiny":
        m = torchvision.models.convnext_tiny(
            weights=torchvision.models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        )
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
        return m

    if name == "vgg16":
        m = torchvision.models.vgg16(
            weights=torchvision.models.VGG16_Weights.DEFAULT if pretrained else None
        )
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        return m

    raise ValueError(f"Unknown model: {name}")

def make_feature_extractor(model: nn.Module, model_name: str):
    model.eval()
    hooks = {}

    if isinstance(model, torchvision.models.ResNet):
        target = model.avgpool
        feat_dim = model.fc.in_features
        def fwd(_, __, out):
            hooks["feat"] = torch.flatten(out, 1)

    elif isinstance(model, torchvision.models.DenseNet):
        target = model.features
        feat_dim = model.classifier.in_features
        def fwd(_, __, out):
            out_ = F.relu(out, inplace=False)
            out_ = F.adaptive_avg_pool2d(out_, (1, 1))
            hooks["feat"] = torch.flatten(out_, 1)

    elif isinstance(model, torchvision.models.MobileNetV2):
        target = model.features
        feat_dim = model.last_channel
        def fwd(_, __, out):
            out_ = F.adaptive_avg_pool2d(out, (1, 1))
            hooks["feat"] = torch.flatten(out_, 1)

    elif isinstance(model, torchvision.models.ConvNeXt):
        target = model.features
        feat_dim = model.classifier[2].in_features
        def fwd(_, __, out):
            out_ = model.avgpool(out)
            hooks["feat"] = torch.flatten(out_, 1)

    elif isinstance(model, torchvision.models.VGG):
        target = model.avgpool
        feat_dim = model.classifier[0].in_features
        def fwd(_, __, out):
            out_ = F.adaptive_avg_pool2d(out, (7, 7))
            hooks["feat"] = torch.flatten(out_, 1)

    else:
        raise ValueError("No feature extractor rule for this model.")

    target.register_forward_hook(fwd)

    def extractor(x: torch.Tensor) -> torch.Tensor:
        hooks.clear()
        _ = model(x)
        return hooks["feat"]

    return extractor, feat_dim

@torch.no_grad()
def extract_features(model: nn.Module, loader: DataLoader, device: torch.device, model_name: str):
    model.eval()
    extractor, _ = make_feature_extractor(model, model_name)
    feats = []
    paths = []
    for (x, _, p) in loader:
        x = x.to(device)
        f = extractor(x).detach().cpu().numpy()
        feats.append(f)
        paths += list(p)
    feats = np.vstack(feats)
    return feats, paths


In [ ]:

# ====================================================================
# SECTION 5: TRAINING, EVALUATION, CALIBRATION & PLOTS
# ====================================================================

@torch.no_grad()
def compute_ece(probs: torch.Tensor, y_true, n_bins: int = 15) -> float:
    confidences, preds = torch.max(probs, dim=1)
    confidences = confidences.detach().cpu().numpy()
    preds = preds.detach().cpu().numpy()
    y_true = np.asarray(y_true)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi)
        if not np.any(mask):
            continue
        acc_bin = (preds[mask] == y_true[mask]).mean()
        conf_bin = confidences[mask].mean()
        ece += (mask.mean()) * abs(acc_bin - conf_bin)
    return float(ece)

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / torch.exp(self.log_temp)

def fit_temperature(logits: torch.Tensor, labels, lr: float = 1e-2) -> TemperatureScaler:
    device = logits.device
    labels = torch.as_tensor(labels, device=device)
    ts = TemperatureScaler().to(device)
    opt = torch.optim.LBFGS(ts.parameters(), lr=lr, max_iter=50)
    nll = nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad(set_to_none=True)
        loss = nll(ts(logits), labels)
        loss.backward()
        return loss
    opt.step(closure)
    return ts

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    loss_fn,
    limit: float = 1.0,
    mixup: float = 0.0,
    label_smoothing: float = 0.0,
) -> float:
    model.train()
    max_steps = int(math.ceil(len(loader) * limit))
    run_loss = 0.0
    steps = 0
    for b, (x, y, _) in enumerate(loader):
        if b >= max_steps:
            break
        x, y = to_device(x, device), to_device(y, device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)

        if label_smoothing > 0:
            ncls = logits.shape[1]
            y_onehot = F.one_hot(y, ncls).float()
            y_smooth = (1 - label_smoothing) * y_onehot + label_smoothing / ncls
            loss = torch.mean(
                torch.sum(-y_smooth * F.log_softmax(logits, 1), dim=1)
            )
        else:
            loss = loss_fn(logits, y)

        loss.backward()
        optimizer.step()
        run_loss += float(loss.item())
        steps += 1

    return run_loss / max(1, steps)

@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    limit: float = 1.0,
    tta: bool = False,
):
    model.eval()
    max_steps = int(math.ceil(len(loader) * limit))
    all_probs = []
    all_targets = []
    all_preds = []
    all_paths = []

    for b, (x, y, p) in enumerate(loader):
        if b >= max_steps:
            break
        x = x.to(device)
        if tta:
            logits = (model(x) + model(torch.flip(x, dims=[-1]))) / 2.0
        else:
            logits = model(x)

        probs = F.softmax(logits, 1)
        preds = probs.argmax(1)

        all_probs.append(probs.cpu())
        all_preds.append(preds.cpu())
        all_targets.append(y)
        all_paths += list(p)

    probs = torch.cat(all_probs, 0).numpy()
    preds = torch.cat(all_preds, 0).numpy()
    targets = torch.cat(all_targets, 0).numpy()

    ncls = probs.shape[1]
    try:
        auroc = roc_auc_score(
            targets,
            probs[:, 1] if ncls == 2 else probs,
            multi_class="ovr" if ncls > 2 else "raise",
        )
    except Exception:
        auroc = float("nan")

    try:
        aucpr = average_precision_score(
            targets,
            probs[:, 1] if ncls == 2 else probs,
            average="macro",
        )
    except Exception:
        aucpr = float("nan")

    f1m = f1_score(targets, preds, average="macro")
    acc = accuracy_score(targets, preds)
    ece = compute_ece(torch.tensor(probs), targets, n_bins=15)

    return {
        "probs": probs,
        "preds": preds,
        "targets": targets,
        "paths": all_paths,
        "auroc": auroc,
        "aucpr": aucpr,
        "f1m": f1m,
        "acc": acc,
        "ece": ece,
    }

# --- Visualization helpers (curves, confusion, calibration, reports) ---------

def plot_learning_curves(history: dict, out_dir, model_name: str) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(6, 4), dpi=200)
    plt.plot(epochs, history["train_loss"], marker="o", label="Train loss")
    plt.plot(epochs, history["val_metric"], marker="s", label="Val metric (AUROC/F1)")
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.title(f"Learning Curves – {model_name}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    save_fig(out_dir / f"{model_name}_learning_curves")

def plot_confusion(y_true, y_pred, classes, out_dir, tag: str) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    cm_sum = cm.sum(axis=1, keepdims=True)
    cm_norm = cm.astype("float") / np.maximum(cm_sum, 1)

    fig, ax = plt.subplots(figsize=(5, 4), dpi=200)

    # viridis is colorblind-friendly and perceptually uniform
    im = ax.imshow(cm_norm, interpolation="nearest", cmap="viridis")
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("Normalized frequency", rotation=90, va="center")

    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticklabels(classes)

    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"Confusion Matrix – {tag}")

    thresh = cm_norm.max() / 2.0 if cm_norm.size > 0 else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            norm_val = cm_norm[i, j]
            count = cm[i, j]
            text_color = "white" if norm_val > thresh else "black"
            ax.text(
                j,
                i,
                f"{norm_val:.2f}\n({count})",
                ha="center",
                va="center",
                color=text_color,
                fontsize=7,
            )

    fig.tight_layout()
    save_fig(out_dir / f"{tag}_confusion_matrix")

def plot_roc_pr(y_true, probs, classes, out_dir, tag: str) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    ncls = probs.shape[1]

    # ROC
    plt.figure(figsize=(5, 4), dpi=200)
    if ncls == 2:
        fpr, tpr, _ = roc_curve(y_true, probs[:, 1])
        auc_ = roc_auc_score(y_true, probs[:, 1])
        plt.plot(fpr, tpr, label=f"ROC (AUC={auc_:.3f})")
    else:
        for c in range(ncls):
            fpr, tpr, _ = roc_curve((y_true == c).astype(int), probs[:, c])
            auc_ = roc_auc_score((y_true == c).astype(int), probs[:, c])
            plt.plot(fpr, tpr, label=f"{classes[c]} (AUC={auc_:.3f})")
    plt.plot([0, 1], [0, 1], "k--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves – {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=7)
    save_fig(out_dir / f"{tag}_roc")

    # PR
    plt.figure(figsize=(5, 4), dpi=200)
    if ncls == 2:
        prec, rec, _ = precision_recall_curve(y_true, probs[:, 1])
        ap = average_precision_score(y_true, probs[:, 1])
        plt.plot(rec, prec, label=f"PR (AP={ap:.3f})")
    else:
        for c in range(ncls):
            prec, rec, _ = precision_recall_curve((y_true == c).astype(int), probs[:, c])
            ap = average_precision_score((y_true == c).astype(int), probs[:, c])
            plt.plot(rec, prec, label=f"{classes[c]} (AP={ap:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision–Recall Curves – {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=7)
    save_fig(out_dir / f"{tag}_pr")

def plot_calibration(probs, y_true, out_dir, tag: str, n_bins: int = 15) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    probs = np.asarray(probs)
    y_true = np.asarray(y_true)
    confidences = probs.max(axis=1)
    preds = probs.argmax(axis=1)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    accs = []
    confs = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi)
        if not np.any(mask):
            accs.append(np.nan)
            confs.append(np.nan)
            continue
        acc_bin = (preds[mask] == y_true[mask]).mean()
        conf_bin = confidences[mask].mean()
        accs.append(acc_bin)
        confs.append(conf_bin)

    accs = np.array(accs)
    confs = np.array(confs)

    plt.figure(figsize=(5, 4), dpi=200)
    valid = ~np.isnan(accs)
    plt.plot(confs[valid], accs[valid], marker="o", label="Empirical")
    plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(f"Reliability Diagram – {tag}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    save_fig(out_dir / f"{tag}_calibration")

def save_classification_report(y_true, y_pred, classes, out_path_base) -> None:
    out_path_base = Path(out_path_base)
    out_path_base.parent.mkdir(parents=True, exist_ok=True)

    report = classification_report(
        y_true,
        y_pred,
        target_names=classes,
        digits=3,
        zero_division=0,
    )

    txt_path = out_path_base.with_suffix(".txt")
    with open(txt_path, "w") as f:
        f.write(report)

    md_path = out_path_base.with_suffix(".md")
    with open(md_path, "w") as f:
        f.write(f"# Classification report – {out_path_base.stem}\n\n")
        f.write("```\n")
        f.write(report)
        f.write("\n```")

    print(f"[INFO] Saved classification report to {txt_path} and {md_path}")

# --- Early Stopping -----------------------------------------------------------

class EarlyStopper:
    def __init__(self, patience=5, mode="max"):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.num_bad_epochs = 0
        self.best_state = None
        self.stop = False

    def step(self, metric, model: nn.Module):
        if self.best is None:
            self.best = metric
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.num_bad_epochs = 0
            return

        improved = (metric > self.best) if self.mode == "max" else (metric < self.best)
        if improved:
            self.best = metric
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.num_bad_epochs = 0
        else:
            self.num_bad_epochs += 1
            if self.num_bad_epochs >= self.patience:
                self.stop = True

# --- Simple Grad-CAM gallery & error gallery stubs ---------------------------

def save_gradcam_gallery(model, device, dataset, out_dir, n: int = 8, model_name: str = ""):
    """
    Minimal Grad-CAM gallery: picks n random validation samples and saves overlay.
    For brevity, this uses only heatmaps (no overlay onto original).
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    # To keep script compact, you can implement a richer overlay later.
    # Here we just sample and skip full implementation if not needed.
    # (Optional: reuse GradCAM class defined later.)
    print(f"[INFO] Grad-CAM gallery placeholder for {model_name} (n={n})")
    return

def save_misclassification_gallery(
    paths, targets, preds, classes, out_dir,
    per_class: int = 12, img_size: int = 224, title: str = ""
):
    """
    Simple grid of misclassified images (grouped loosely by class).
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    paths = np.asarray(paths)
    targets = np.asarray(targets)
    preds = np.asarray(preds)

    mis_idx = np.where(targets != preds)[0]
    if len(mis_idx) == 0:
        print(f"[INFO] No misclassifications to plot for {title}.")
        return

    mis_paths = paths[mis_idx][:per_class]
    grid_path = out_dir / f"errors_{title}_grid"
    imgs = []
    for p in mis_paths:
        try:
            im = Image.open(p).convert("RGB").resize((img_size, img_size))
            imgs.append(im)
        except Exception:
            continue
    if not imgs:
        return
    cols = 4
    rows = math.ceil(len(imgs) / cols)
    grid = Image.new("RGB", (cols * img_size, rows * img_size), (255, 255, 255))
    for i, im in enumerate(imgs):
        r, c = divmod(i, cols)
        grid.paste(im, (c * img_size, r * img_size))
    grid.save(str(grid_path.with_suffix(".png")))
    try:
        grid.save(str(grid_path.with_suffix(".pdf")))
    except Exception:
        pass


In [ ]:
# ====================================================================
# SECTION 6: GRAD-CAM AND ATTENTION MASKING
# ====================================================================

def get_cam_target_layer(model: nn.Module) -> nn.Module:
    if isinstance(model, torchvision.models.ResNet):
        blk = model.layer4[-1]
        return blk.conv3 if hasattr(blk, "conv3") and isinstance(blk.conv3, nn.Conv2d) else blk.conv2
    if isinstance(model, torchvision.models.DenseNet):
        convs = [m for m in model.features.modules() if isinstance(m, nn.Conv2d)]
        return convs[-1] if convs else None
    if isinstance(model, torchvision.models.MobileNetV2):
        convs = [m for m in model.features.modules() if isinstance(m, nn.Conv2d)]
        return convs[-1] if convs else None
    if isinstance(model, torchvision.models.ConvNeXt):
        convs = [m for m in model.features.modules() if isinstance(m, nn.Conv2d)]
        return convs[-1] if convs else model.features[-1]
    if isinstance(model, torchvision.models.VGG):
        convs = [m for m in model.features if isinstance(m, nn.Conv2d)]
        return convs[-1] if convs else None
    for m in reversed(list(model.modules())):
        if isinstance(m, nn.Conv2d):
            return m
    raise RuntimeError("No Conv2d layer found as Grad-CAM target.")

def denorm_img_tensor(x: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device)[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225], device=x.device)[:, None, None]
    return (x * std + mean).clamp(0, 1)

class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module, device: torch.device):
        self.model = model
        self.model.eval()
        self.device = device
        self.activations = None
        self.gradients = None
        self._handles = []

        def fwd_hook(_, __, out):
            self.activations = out if isinstance(out, torch.Tensor) else None

        def bwd_hook(_, __, grad_out):
            g = grad_out[0] if isinstance(grad_out, (list, tuple)) else grad_out
            self.gradients = g if isinstance(g, torch.Tensor) else None

        self._handles.append(target_layer.register_forward_hook(fwd_hook))
        try:
            self._handles.append(target_layer.register_full_backward_hook(bwd_hook))
        except AttributeError:
            self._handles.append(target_layer.register_backward_hook(bwd_hook))

    def __call__(self, x: torch.Tensor, class_idx: torch.Tensor | None = None) -> torch.Tensor:
        x = x.to(self.device).requires_grad_(True)
        self.activations = None
        self.gradients = None

        logits = self.model(x)
        class_idx = class_idx if class_idx is not None else logits.argmax(1)
        class_idx = class_idx.to(self.device).view(-1)

        self.model.zero_grad(set_to_none=True)
        loss = logits[torch.arange(x.size(0), device=self.device), class_idx]
        with torch.enable_grad():
            loss.sum().backward()

        if (self.activations is not None) and (self.gradients is not None):
            weights = self.gradients.mean(dim=(2, 3), keepdim=True)
            cam = (weights * self.activations).sum(dim=1, keepdim=True)
            cam = F.relu(cam)
        elif self.activations is not None:
            cam = F.relu(self.activations.sum(dim=1, keepdim=True))
        else:
            B, _, H, W = x.shape
            cam = torch.full(
                (B, 1, max(1, H // 32), max(1, W // 32)),
                1e-6,
                device=self.device,
            )

        B, _, Hc, Wc = cam.shape
        camf = cam.view(B, -1)
        camf = (camf - camf.min(1, keepdim=True).values) / (
            camf.max(1, keepdim=True).values - camf.min(1, keepdim=True).values + 1e-6
        )
        return camf.view(B, 1, Hc, Wc)

    def close(self):
        for h in self._handles:
            try:
                h.remove()
            except Exception:
                pass
        self._handles.clear()

@torch.no_grad()
def make_population_attention_mask(
    model: nn.Module,
    device: torch.device,
    dataset: Dataset,
    out_path,
    model_name: str = "",
    cohort_name: str = "Population",
    class_idx=None,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    target_layer = get_cam_target_layer(model)
    cam_fn = GradCAM(model, target_layer, device)

    sum_cam = None
    N = 0
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)

    for _, (x, y, paths) in enumerate(loader):
        x = x.to(device)
        target_y = to_device(y, device) if class_idx is None else torch.full_like(y, fill_value=class_idx).to(device)
        with torch.enable_grad():
            cam_batch = cam_fn(x, class_idx=target_y)

        try:
            cam_batch = F.interpolate(
                cam_batch,
                size=(x.shape[2], x.shape[3]),
                mode="bilinear",
                align_corners=False,
            )
        except Exception:
            continue

        B = cam_batch.shape[0]
        camf = cam_batch.view(B, -1)
        camf = (camf - camf.min(1, keepdim=True).values) / (
            camf.max(1, keepdim=True).values - camf.min(1, keepdim=True).values + 1e-6
        )
        cam_agg = camf.view(cam_batch.shape).sum(dim=0).cpu().numpy()

        if sum_cam is None:
            sum_cam = cam_agg.squeeze()
        else:
            sum_cam += cam_agg.squeeze()
        N += B

    cam_fn.close()
    if N == 0:
        print("[WARN] No samples processed for attention mask.")
        return None

    pop_mask = sum_cam / N
    plt.figure(figsize=(4, 4), dpi=200)
    im = plt.imshow(pop_mask, cmap="jet", vmin=0.0, vmax=pop_mask.max())
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.axis("off")
    plt.title(f"Attention Mask: {model_name} ({cohort_name})")
    save_fig(out_path)
    print(f"[Attention Mask] Saved population mask for {cohort_name} ({N} images).")
    return pop_mask




In [ ]:
# ====================================================================
# SECTION 7: VISUALIZATION (COMPOSITES & IMAGE ENHANCEMENTS)
# ====================================================================

def _ensure_float01(img_np: np.ndarray) -> np.ndarray:
    if img_np.dtype == np.uint8:
        return img_np.astype(np.float32) / 255.0
    return img_np.astype(np.float32)

def rgb_to_lab_np(img_np: np.ndarray) -> np.ndarray:
    x = _ensure_float01(img_np)
    if rgb2lab is not None:
        return rgb2lab(x).astype(np.float32)
    if "cv2" in sys.modules:
        import cv2
        x8 = (x * 255.0).astype(np.uint8)
        return cv2.cvtColor(x8, cv2.COLOR_RGB2LAB).astype(np.float32)
    return np.stack(
        [x.mean(axis=-1) * 100.0, np.zeros_like(x[..., 0]), np.zeros_like(x[..., 0])],
        axis=-1,
    )

def lab_to_rgb_np(lab_np: np.ndarray) -> np.ndarray:
    if lab2rgb is not None:
        return np.clip(lab2rgb(lab_np).astype(np.float32), 0.0, 1.0)
    if "cv2" in sys.modules:
        import cv2
        lab_u8 = np.zeros_like(lab_np, dtype=np.uint8)
        lab_u8[..., 0] = np.clip(lab_np[..., 0] * (255.0 / 100.0), 0, 255)
        lab_u8[..., 1:] = np.clip(lab_np[..., 1:] + 128.0, 0, 255)
        rgb = cv2.cvtColor(lab_u8, cv2.COLOR_LAB2RGB).astype(np.float32) / 255.0
        return np.clip(rgb, 0.0, 1.0)
    return np.clip(lab_np[..., 0] / 100.0, 0.0, 1.0)

def enhance_image_rgb(img_pil: Image.Image) -> Image.Image:
    arr = np.array(img_pil.convert("RGB"))
    lab = rgb_to_lab_np(arr)
    if lab.shape[-1] < 3:
        return img_pil
    L, a, b = lab[..., 0], lab[..., 1], lab[..., 2]
    L01 = np.clip(L / 100.0, 0, 1)
    if exposure is not None and hasattr(exposure, "equalize_adapthist"):
        L_eq = exposure.equalize_adapthist(L01, clip_limit=0.01, nbins=256)
    else:
        L_eq = np.clip(
            (L01 - L01.min()) / (L01.max() - L01.min() + 1e-6),
            0,
            1,
        )
    L_new = np.clip(L_eq * 100.0, 0, 100)
    lab2 = np.stack([L_new, a, b], axis=-1).astype(np.float32)
    rgb = lab_to_rgb_np(lab2)
    if restoration is not None and hasattr(restoration, "denoise_bilateral"):
        try:
            rgb = restoration.denoise_bilateral(rgb, channel_axis=-1)
        except TypeError:
            rgb = restoration.denoise_bilateral(rgb, multichannel=True)
    out = (np.clip(rgb, 0, 1) * 255).astype(np.uint8)
    return Image.fromarray(out)

def save_composite_grid(
    img_paths,
    out_path,
    cols: int = 4,
    size: int = 224,
    enhance: bool = True,
    title: str | None = None,
):
    imgs = []
    for p in img_paths:
        try:
            im = Image.open(p).convert("RGB").resize((size, size))
            if enhance:
                im = enhance_image_rgb(im)
            imgs.append(im)
        except Exception:
            continue
    if not imgs:
        return

    rows = math.ceil(len(imgs) / cols)
    grid = Image.new("RGB", (cols * size, rows * size), (255, 255, 255))
    for i, im in enumerate(imgs):
        r, c = divmod(i, cols)
        grid.paste(im, (c * size, r * size))

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    grid.save(str(out_path.with_suffix(".png")))
    try:
        grid.save(str(out_path.with_suffix(".pdf")))
    except Exception:
        pass

In [ ]:

# ====================================================================
# SECTION 8: COHORT MEAN FACE AND ALIGNMENT
# ====================================================================

def _lower(s):
    return str(s).lower()

def _oval_mask_rgb(img: Image.Image, feather: float = 0.04) -> Image.Image:
    w, h = img.size
    mask = Image.new("L", (w, h), 0)
    d = ImageDraw.Draw(mask)
    inset = int(min(w, h) * 0.06)
    d.ellipse((inset, inset, w - inset, h - inset), fill=255)
    if feather > 0:
        r = max(1, int(min(w, h) * feather))
        mask = mask.filter(ImageFilter.GaussianBlur(radius=r))
    out = Image.new("RGBA", (w, h))
    out.paste(img, (0, 0), mask)
    return out.convert("RGB")

def _try_import_mediapipe():
    return mp if "mediapipe" in sys.modules else None

def _align_face_mediapipe(pil_rgb: Image.Image, out_size: int, mp_module):
    if mp_module is None or "cv2" not in sys.modules:
        return None
    import cv2
    img = np.array(pil_rgb)
    mpf = mp_module.solutions.face_mesh
    with mpf.FaceMesh(
        static_image_mode=True,
        refine_landmarks=True,
        max_num_faces=1
    ) as fm:
        res = fm.process(img[:, :, ::-1])
    if not res.multi_face_landmarks:
        return None
    lm = res.multi_face_landmarks[0].landmark
    H, W = img.shape[:2]
    left_eye_idx = [33, 133, 160, 159, 158, 157, 173]
    right_eye_idx = [263, 362, 385, 386, 387, 373, 380]
    mouth_idx = [13, 14, 78, 308, 82, 312, 87, 317]

    def mean_xy(idxs):
        xs = [lm[i].x * W for i in idxs]
        ys = [lm[i].y * H for i in idxs]
        return (sum(xs) / len(xs), sum(ys) / len(ys))

    Lx, Ly = mean_xy(left_eye_idx)
    Rx, Ry = mean_xy(right_eye_idx)
    Mx, My = mean_xy(mouth_idx)
    S = out_size
    src = np.float32([[Lx, Ly], [Rx, Ry], [Mx, My]])
    tgt = np.float32([[0.35 * S, 0.40 * S], [0.65 * S, 0.40 * S], [0.50 * S, 0.70 * S]])
    M = cv2.getAffineTransform(src, tgt)
    aligned = cv2.warpAffine(
        img,
        M,
        (S, S),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT,
    )
    return Image.fromarray(aligned)

def _align_face_center(pil_rgb: Image.Image, out_size: int) -> Image.Image:
    w, h = pil_rgb.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    crop = pil_rgb.crop((left, top, left + side, top + side))
    return crop.resize((out_size, out_size), Image.BILINEAR)

def _accumulate_mean(
    paths,
    out_size: int,
    use_mp: bool = True,
    mask_oval: bool = True,
    max_n: int | None = None,
):
    mp_module = _try_import_mediapipe() if use_mp else None
    S = out_size
    acc = np.zeros((S, S, 3), dtype=np.float64)
    n = 0
    for p in paths:
        if max_n and n >= max_n:
            break
        try:
            im = Image.open(p).convert("RGB")
            alg = _align_face_mediapipe(im, S, mp_module) if mp_module is not None else _align_face_center(im, S)
            if alg is None and mp_module is not None:
                alg = _align_face_center(im, S)
            if alg is None:
                continue
            acc += np.asarray(alg, dtype=np.float64)
            n += 1
        except Exception:
            continue
    if n == 0:
        return None, 0
    mean = np.clip(acc / n, 0, 255).astype(np.uint8)
    out = Image.fromarray(mean)
    if mask_oval:
        out = _oval_mask_rgb(out)
    return out, n

def _score_mse(img_arr, ref_arr):
    diff = img_arr.astype(np.float32) - ref_arr.astype(np.float32)
    return float(np.mean(diff * diff))

def _score_ssim(img_arr, ref_arr):
    try:
        from skimage.metrics import structural_similarity as ssim
        s, _ = ssim(img_arr, ref_arr, channel_axis=-1, full=True)
        return 1.0 - float(s)
    except Exception:
        return _score_mse(img_arr, ref_arr)

def _score_func(name: str):
    return _score_mse if name.lower() == "mse" else _score_ssim

def _align_single_for_scoring(path, S: int, use_mp: bool):
    mp_module = _try_import_mediapipe() if use_mp else None
    try:
        im = Image.open(path).convert("RGB")
        alg = _align_face_mediapipe(im, S, mp_module) if mp_module is not None else _align_face_center(im, S)
        if alg is None and mp_module is not None:
            alg = _align_face_center(im, S)
        if alg is None:
            return None
        return np.asarray(alg, dtype=np.uint8)
    except Exception:
        return None

def make_composites_and_pairwise(
    trainval_df: pd.DataFrame,
    test_df: pd.DataFrame,
    class_to_id: dict,
    cfg: dict,
    project_dir: Path,
):
    out_root = project_dir / "artifacts" / "mean_faces"
    pair_root = out_root / "pairwise"
    pair_root.mkdir(parents=True, exist_ok=True)
    out_root.mkdir(parents=True, exist_ok=True)

    cohorts = cfg["mean_face"]["cohorts"]
    S = int(cfg["mean_face"]["img_size"])
    use_mp = (_lower(cfg["mean_face"].get("align", "mediapipe")) == "mediapipe")
    mask_oval = bool(cfg["mean_face"]["mask_oval"])
    max_n = cfg["mean_face"]["max_per_cohort"]
    use_splits = cfg["mean_face"].get("splits", ["train", "valid", "test"])

    ds_label = next(
        (v for k, v in class_to_id.items()
         if k.lower() in ("downsyndrome", "down", "trisomy21", "t21", "ds")),
        None,
    )
    if ds_label is None:
        print("[Composite] No 'downsyndrome' class found; skipping.")
        return

    df_all = pd.concat([trainval_df, test_df], axis=0, ignore_index=True)
    df_all = df_all[df_all["split"].isin(use_splits)].copy()
    ds_df = df_all[df_all["label"] == ds_label].copy()
    if ds_df.empty:
        print("[Composite] No down-syndrome images; skipping.")
        return

    ds_df["cohort_inferred"] = ds_df["path"].map(infer_cohort_from_path)
    paths_by = {
        c: ds_df.loc[
            ds_df["cohort_inferred"].str.lower() == c.lower(), "path"
        ].tolist()
        for c in cohorts
    }

    composites = {}
    tiles = []
    pad = 30

    for coh in cohorts:
        paths = paths_by.get(coh, [])
        comp_img, used = _accumulate_mean(
            paths,
            out_size=S,
            use_mp=use_mp,
            mask_oval=mask_oval,
            max_n=max_n,
        )
        if comp_img is None:
            continue
        comp_path = out_root / f"mean_{coh.lower()}.png"
        comp_img.save(comp_path)
        composites[coh] = np.asarray(comp_img, dtype=np.uint8)
        tiles.append((coh, comp_img))

    if tiles:
        tile_w, tile_h = tiles[0][1].size
        panel = Image.new(
            "RGB",
            (pad + len(tiles) * (tile_w + pad), tile_h + 2 * pad),
            (0, 0, 0),
        )
        x = pad
        for name, im in tiles:
            panel.paste(im, (x, pad))
            x += tile_w + pad
        panel.save(out_root / "panel_downsyndrome_cohorts.png")
        try:
            panel.save(out_root / "panel_downsyndrome_cohorts.pdf")
        except Exception:
            pass
        print(f"[Composite] Saved panel with {len(tiles)} cohorts.")
    else:
        print("[Composite] No composites generated; comparisons skipped.")
        return

    # Pairwise & confusion logic can be added here following your original approach.


In [ ]:
# ====================================================================
# SECTION 9: CLUSTERING AND ASSOCIATION ANALYSIS
# ====================================================================

def cohort_associations(labels_csv, meta_df: pd.DataFrame, out_md):
    from scipy.stats import chi2_contingency
    lab = pd.read_csv(labels_csv)
    meta = meta_df.reset_index(drop=True).copy()
    meta["cohort"] = meta["path"].map(infer_cohort_from_path)
    meta = meta.join(lab["cluster"])
    tab = pd.crosstab(meta["cluster"], meta["cohort"])
    chi2, p, dof, _ = chi2_contingency(tab)

    out_md = Path(out_md)
    with open(out_md, "w") as f:
        f.write("# Cohort Association (Clusters vs Cohort)\n\n")
        f.write(f"Chi2={chi2:.2f}, dof={dof}, p={p:.3e}\n\n")
        f.write("Contingency table:\n\n")
        f.write(tab.to_string())
        f.write("\n")

    return {"chi2": chi2, "p": p, "dof": dof, "table": tab}

def _stacked_bar_clusters_by_cohort(meta_df: pd.DataFrame, outpath):
    tab = pd.crosstab(meta_df["cluster"], meta_df["cohort"])
    tab = tab.div(tab.sum(axis=0).replace(0, 1), axis=1)
    plt.figure(figsize=(6, 3.2), dpi=200)
    bottom = np.zeros(tab.shape[1])
    for ridx, row in tab.iterrows():
        plt.bar(
            range(tab.shape[1]),
            row.values,
            bottom=bottom,
            label=f"C{ridx}",
        )
        bottom += row.values
    plt.xticks(range(tab.shape[1]), tab.columns, rotation=30, ha="right")
    plt.ylabel("Fraction within cohort")
    plt.title("Cluster composition by cohort")
    plt.legend(ncol=min(4, tab.shape[0]))
    save_fig(outpath)

def run_clustering(
    project_dir: Path,
    model_name: str,
    val_df: pd.DataFrame,
    n_components: int = 50,
    k_candidates = [2, 3, 4, 5, 6],
    cfg: dict | None = None,
):
    device = DEVICE
    mpath = project_dir / "artifacts" / "models" / model_name / "best_fold0.pt"
    if not mpath.exists():
        print(f"[WARN] Best checkpoint not found for {model_name}; skipping clustering.")
        return {"k": None, "silhouette": None, "assoc_p": None}

    model = make_model(
        model_name,
        num_classes=val_df["label"].nunique(),
        pretrained=False,
    ).to(device)
    disable_inplace_relu(model)
    model.load_state_dict(torch.load(mpath, map_location=device))

    val_loader = DataLoader(
        ImageCSV(val_df, img_size=DEFAULT_CFG["img_size"], train=False),
        batch_size=64,
        shuffle=False,
        num_workers=2,
    )
    feats, paths = extract_features(model, val_loader, device, model_name)
    Z = StandardScaler().fit_transform(feats)
    pca = PCA(n_components=min(n_components, Z.shape[1]))
    Zp = pca.fit_transform(Z)

    best_k, best_s = None, -1
    for k in k_candidates:
        km_tmp = KMeans(n_clusters=k, n_init=10, random_state=DEFAULT_CFG["seed"]).fit(Zp)
        s = silhouette_score(Zp, km_tmp.labels_) if k > 1 else -1
        if s > best_s:
            best_s, best_k = s, k

    km = KMeans(n_clusters=best_k, n_init=10, random_state=DEFAULT_CFG["seed"]).fit(Zp)
    labels = km.labels_

    overall_dir = project_dir / "artifacts" / "clustering" / "overall"
    out_csv = overall_dir / f"labels_{model_name}_k{best_k}.csv"
    pd.DataFrame({"path": paths, "cluster": labels}).to_csv(out_csv, index=False)

    meta_overall = pd.DataFrame({"path": paths, "cluster": labels})
    meta_overall["cohort"] = meta_overall["path"].map(infer_cohort_from_path)
    assoc = cohort_associations(out_csv, meta_overall[["path"]], project_dir / "reports" / f"clustering_overall_{model_name}.md")
    _stacked_bar_clusters_by_cohort(meta_overall, overall_dir / f"{model_name}_clusters_by_cohort")

    print(
        f"[Clustering-Overall] best_k={best_k}, "
        f"silhouette={best_s:.3f}, chi2_p={assoc['p']:.3e}"
    )
    return {"k": best_k, "silhouette": best_s, "assoc_p": assoc["p"]}




In [ ]:
# ====================================================================
# SECTION 10: TRAIN & SELECT
# ====================================================================

def train_and_select(
    models,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    class_to_id: dict,
    cfg: dict,
    project_dir: Path,
):
    device = DEVICE
    classes = [k for k, _ in sorted(class_to_id.items(), key=lambda kv: kv[1])]
    train_ds, val_ds, train_loader, val_loader = make_dataloaders(train_df, val_df, cfg)
    results = []

    artifacts_root = project_dir / "artifacts"
    figures_root = artifacts_root / "figures"
    models_root = artifacts_root / "models"

    cw = None
    if cfg["training"]["class_weights"]:
        counts = (
            train_df["label"].value_counts().sort_index().values.astype(np.float32)
        )
        weights = counts.sum() / (counts + 1e-6)
        cw = torch.from_numpy(weights / weights.mean()).float().to(device)

    for mname in models:
        print(f"\n=== Model: {mname} ===")
        model = make_model(mname, num_classes=len(classes), pretrained=True).to(device)
        disable_inplace_relu(model)

        loss_fn = nn.CrossEntropyLoss(weight=cw)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg["training"]["lr"],
            weight_decay=cfg["training"]["weight_decay"],
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, cfg["training"]["max_epochs"]),
        )

        stopper = EarlyStopper(patience=cfg["training"]["patience"], mode="max")
        history = {"train_loss": [], "val_metric": []}
        best_epoch = -1
        best_val = -np.inf

        for epoch in range(1, cfg["training"]["max_epochs"] + 1):
            t0 = time.time()
            tr_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                device,
                loss_fn,
                limit=cfg["training"]["limit_train_batches"],
                mixup=cfg["training"]["mixup"],
                label_smoothing=cfg["training"]["label_smoothing"],
            )
            val_out = evaluate(
                model,
                val_loader,
                device,
                limit=cfg["training"]["limit_val_batches"],
                tta=cfg["tta"],
            )
            primary = (
                val_out["auroc"] if not np.isnan(val_out["auroc"])
                else val_out["f1m"]
            )

            history["train_loss"].append(tr_loss)
            history["val_metric"].append(primary)

            scheduler.step()
            stopper.step(primary, model)

            dt = time.time() - t0
            print(
                f"Epoch {epoch}: loss={tr_loss:.4f} "
                f"val_auroc={val_out['auroc']:.4f} "
                f"f1m={val_out['f1m']:.4f} "
                f"acc={val_out['acc']:.4f} "
                f"ece={val_out['ece']:.4f} "
                f"time={dt:.1f}s"
            )

            if primary > best_val:
                best_val = primary
                best_epoch = epoch
                wdir = models_root / mname
                wdir.mkdir(parents=True, exist_ok=True)
                torch.save(model.state_dict(), wdir / "best_fold0.pt")

            if stopper.stop:
                print(f"[INFO] Early stopping at epoch {epoch}.")
                break

        if stopper.best_state is not None:
            model.load_state_dict(stopper.best_state)

        calib_note = "off"
        if cfg["calibrate"]:
            model.eval()
            all_logits = []
            all_y = []
            for (x, y, _) in val_loader:
                x = x.to(device)
                all_logits.append(model(x).detach().cpu())
                all_y.append(y)
            all_logits = torch.cat(all_logits, 0).to(device)
            all_y = torch.cat(all_y, 0).numpy()
            scaler = fit_temperature(all_logits, all_y)

            class ModelWithTemp(nn.Module):
                def __init__(self, base, scaler_):
                    super().__init__()
                    self.base = base
                    self.scaler = scaler_
                def forward(self, x):
                    return self.scaler(self.base(x))

            model = ModelWithTemp(model, scaler).to(device)
            calib_note = "temperature"

        # Plots & galleries
        plot_learning_curves(history, figures_root / mname, mname)
        val_out = evaluate(model, val_loader, device, limit=1.0, tta=cfg["tta"])
        plot_confusion(
            val_out["targets"], val_out["preds"], classes,
            figures_root / mname, f"{mname}_val"
        )
        plot_roc_pr(
            val_out["targets"], val_out["probs"], classes,
            figures_root / mname, f"{mname}_val"
        )
        plot_calibration(
            val_out["probs"], val_out["targets"],
            figures_root / mname, f"{mname}_val"
        )
        save_gradcam_gallery(
            model, device, val_ds, figures_root / mname / "gradcam",
            n=cfg["viz"]["gradcam_n"], model_name=mname
        )
        save_misclassification_gallery(
            val_out["paths"], val_out["targets"], val_out["preds"],
            classes, figures_root / mname / "errors",
            per_class=12, img_size=cfg["img_size"], title=mname
        )

        # Classification report
        save_classification_report(
            val_out["targets"],
            val_out["preds"],
            classes,
            project_dir / "reports" / f"classification_report_{mname}"
        )

        results.append({
            "model": mname,
            "fold": 0,
            "auroc": val_out["auroc"],
            "aucpr": val_out["aucpr"],
            "f1m": val_out["f1m"],
            "acc": val_out["acc"],
            "ece": val_out["ece"],
            "calibration": calib_note,
        })

        with open(project_dir / "reports" / f"model_card_{mname}.md", "w") as f:
            f.write(f"# Model Card: {mname}\n\n")
            f.write("- Primary metric: AUROC (robust to class imbalance)\n")
            f.write(
                "- Validation metrics: "
                f"AUROC={val_out['auroc']:.4f}, "
                f"AUC-PR={val_out['aucpr']:.4f}, "
                f"F1m={val_out['f1m']:.4f}, "
                f"Acc={val_out['acc']:.4f}, "
                f"ECE={val_out['ece']:.4f}\n"
            )

    res_df = pd.DataFrame(results)
    metrics_path = project_dir / "reports" / "metrics.csv"
    res_df.to_csv(metrics_path, index=False)
    print(f"[INFO] Saved metrics table to {metrics_path}")

    res_df = res_df.sort_values("auroc", ascending=False)
    top3 = res_df["model"].unique()[:3].tolist()

    top_dir = project_dir / "artifacts" / "models" / "top3"
    top_dir.mkdir(parents=True, exist_ok=True)
    for m in top3:
        src = project_dir / "artifacts" / "models" / m / "best_fold0.pt"
        if src.exists():
            shutil.copy2(src, top_dir / f"{m}_best.pt")

    print("\nTop-3 architectures by AUROC:", top3)
    return res_df, top3



In [ ]:
# ====================================================================
# SECTION 11: ORCHESTRATION
# ====================================================================

def run_pipeline(cfg: dict):
    set_all_seeds(cfg["seed"])
    project_dir = ensure_project_tree(cfg["project_dir"])
    print(f"[INFO] Project root: {project_dir}")
    print(f"[INFO] Dataset root: {cfg['dataset_root']}")

    trainval, testdf, class_to_id = discover_dataset(cfg["dataset_root"])
    trainval.to_csv(project_dir / "data" / "labels_trainval.csv", index=False)
    testdf.to_csv(project_dir / "data" / "labels_test.csv", index=False)
    pd.DataFrame(
        sorted(class_to_id.items(), key=lambda kv: kv[1]),
        columns=["class", "id"],
    ).to_csv(project_dir / "data" / "class_to_id.csv", index=False)

    if cfg.get("quick_mode", False):
        print("⚡ Quick mode ON: reducing data for fast debugging.")
        train_df_q = trainval[trainval.split == "train"]
        val_df_q = trainval[trainval.split == "valid"]

        def head_strat(df, per_class):
            return df.sort_values("path").groupby("label", group_keys=False).head(per_class)

        trainval = pd.concat(
            [head_strat(train_df_q, 400), head_strat(val_df_q, 80)],
            axis=0,
        )
        cfg["training"]["max_epochs"] = min(5, cfg["training"]["max_epochs"])
        cfg["training"]["limit_train_batches"] = 0.5
        cfg["training"]["limit_val_batches"] = 0.5

    train_df = trainval[trainval["split"] == "train"].copy()
    val_df   = trainval[trainval["split"] == "valid"].copy()

    print("Class distribution by split:\n", trainval.groupby(["split", "class_name"]).size())

    res_df, top3 = train_and_select(cfg["models"], train_df, val_df, class_to_id, cfg, project_dir)

    if len(top3) > 0:
        best_model = top3[0]
        device = DEVICE
        best_model_path = project_dir / "artifacts" / "models" / best_model / "best_fold0.pt"
        model = make_model(best_model, num_classes=trainval["label"].nunique(), pretrained=False).to(device)
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        model.eval()

        ds_class_id = next(
            (v for k, v in class_to_id.items()
             if k.lower() in ("downsyndrome", "down", "trisomy21", "t21", "ds")),
            None,
        )

        all_data_df = pd.concat([trainval, testdf], axis=0, ignore_index=True)
        all_data_df["cohort_inferred"] = all_data_df["path"].map(infer_cohort_from_path)

        attention_mask_dir = project_dir / "artifacts" / "figures" / best_model / "attention_masks"
        attention_mask_dir.mkdir(parents=True, exist_ok=True)

        for cohort in cfg["mean_face"]["cohorts"]:
            cohort_df = all_data_df[
                (all_data_df["cohort_inferred"].str.lower() == cohort.lower())
                & (all_data_df["label"] == ds_class_id)
            ].copy()
            if len(cohort_df) < 50:
                print(f"[WARN] Cohort {cohort} has only {len(cohort_df)} DS samples; skipping attention mask.")
                continue
            cohort_ds = ImageCSV(cohort_df, img_size=cfg["img_size"], train=False)
            make_population_attention_mask(
                model=model,
                device=device,
                dataset=cohort_ds,
                out_path=attention_mask_dir / f"mask_ds_{cohort.lower()}",
                model_name=best_model,
                cohort_name=cohort,
                class_idx=ds_class_id,
            )

    if len(top3) > 0 and cfg["clustering"].get("overall_enabled", True):
        run_clustering(
            project_dir,
            top3[0],
            val_df,
            n_components=cfg["clustering"]["n_components_pca"],
            k_candidates=cfg["clustering"]["k_candidates"],
            cfg=cfg,
        )

    if cfg.get("mean_face", {}).get("enabled", False) and cfg.get("mean_face_compare", {}).get("enabled", False):
        make_composites_and_pairwise(trainval, testdf, class_to_id, cfg, project_dir)

    print("\n[INFO] Pipeline execution complete.")
    print(f"- All outputs saved to: {project_dir}")
    return {"results": res_df, "top3": top3}


In [ ]:
# ====================================================================
# ENTRY POINT
# ====================================================================

if __name__ == "__main__":
    out = run_pipeline(DEFAULT_CFG)


[INFO] Project root: /content/drive/MyDrive/biomed_pipeline_updated_run
[INFO] Dataset root: /content/drive/MyDrive/t21_dataset1
[Probe] train: dir=/content/drive/MyDrive/t21_dataset1/train | class_dirs=0 | imgs_found=6047
[Probe] valid: dir=/content/drive/MyDrive/t21_dataset1/valid | class_dirs=0 | imgs_found=599
[Probe] test: dir=/content/drive/MyDrive/t21_dataset1/test | class_dirs=0 | imgs_found=310

[Discovery Summary]
split  class_name     
train  downsyndrome       2941
       nondownsyndrome    3106
valid  downsyndrome        296
       nondownsyndrome     303
dtype: int64
Class distribution by split:
 split  class_name     
train  downsyndrome       2941
       nondownsyndrome    3106
valid  downsyndrome        296
       nondownsyndrome     303
dtype: int64

=== Model: mobilenet_v2 ===
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 231MB/s]


Epoch 1: loss=0.2494 val_auroc=0.9946 f1m=0.9465 acc=0.9466 ece=0.0145 time=464.6s
Epoch 2: loss=0.0700 val_auroc=0.9926 f1m=0.9666 acc=0.9666 ece=0.0184 time=33.7s
Epoch 3: loss=0.0384 val_auroc=0.9946 f1m=0.9699 acc=0.9699 ece=0.0121 time=34.4s
Epoch 4: loss=0.0240 val_auroc=0.9953 f1m=0.9716 acc=0.9716 ece=0.0153 time=33.4s
Epoch 5: loss=0.0119 val_auroc=0.9959 f1m=0.9749 acc=0.9750 ece=0.0188 time=33.1s
Epoch 6: loss=0.0067 val_auroc=0.9945 f1m=0.9766 acc=0.9766 ece=0.0185 time=34.3s
Epoch 7: loss=0.0045 val_auroc=0.9958 f1m=0.9783 acc=0.9783 ece=0.0186 time=33.6s
Epoch 8: loss=0.0045 val_auroc=0.9947 f1m=0.9766 acc=0.9766 ece=0.0156 time=34.2s
[INFO] Early stopping at epoch 8.
[INFO] Grad-CAM gallery placeholder for mobilenet_v2 (n=8)
[INFO] Saved classification report to /content/drive/MyDrive/biomed_pipeline_updated_run/reports/classification_report_mobilenet_v2.txt and /content/drive/MyDrive/biomed_pipeline_updated_run/reports/classification_report_mobilenet_v2.md

=== Model: d

100%|██████████| 30.8M/30.8M [00:00<00:00, 240MB/s]


Epoch 1: loss=0.1779 val_auroc=0.9927 f1m=0.9666 acc=0.9666 ece=0.0134 time=64.2s
Epoch 2: loss=0.0546 val_auroc=0.9939 f1m=0.9682 acc=0.9683 ece=0.0231 time=63.5s
Epoch 3: loss=0.0579 val_auroc=0.9960 f1m=0.9816 acc=0.9816 ece=0.0093 time=63.7s
Epoch 4: loss=0.0209 val_auroc=0.9957 f1m=0.9816 acc=0.9816 ece=0.0129 time=63.5s
Epoch 5: loss=0.0083 val_auroc=0.9956 f1m=0.9816 acc=0.9816 ece=0.0133 time=63.5s
Epoch 6: loss=0.0043 val_auroc=0.9963 f1m=0.9883 acc=0.9883 ece=0.0141 time=63.5s
Epoch 7: loss=0.0021 val_auroc=0.9980 f1m=0.9883 acc=0.9883 ece=0.0074 time=63.6s
Epoch 8: loss=0.0014 val_auroc=0.9975 f1m=0.9917 acc=0.9917 ece=0.0098 time=63.7s
Epoch 9: loss=0.0012 val_auroc=0.9972 f1m=0.9917 acc=0.9917 ece=0.0083 time=64.0s
Epoch 10: loss=0.0011 val_auroc=0.9975 f1m=0.9917 acc=0.9917 ece=0.0098 time=63.7s
[INFO] Early stopping at epoch 10.
[INFO] Grad-CAM gallery placeholder for densenet121 (n=8)
[INFO] Saved classification report to /content/drive/MyDrive/biomed_pipeline_updated_r

...

Epoch 1: loss=0.2158 val_auroc=0.9893 f1m=0.9449 acc=0.9449 ece=0.0162 time=70.8s
Epoch 2: loss=0.0659 val_auroc=0.9940 f1m=0.9666 acc=0.9666 ece=0.0257 time=70.4s
Epoch 3: loss=0.0376 val_auroc=0.9921 f1m=0.9616 acc=0.9616 ece=0.0192 time=71.1s
Epoch 4: loss=0.0180 val_auroc=0.9969 f1m=0.9816 acc=0.9816 ece=0.0128 time=70.5s
Epoch 5: loss=0.0066 val_auroc=0.9988 f1m=0.9800 acc=0.9800 ece=0.0180 time=71.6s
Epoch 6: loss=0.0067 val_auroc=0.9968 f1m=0.9800 acc=0.9800 ece=0.0190 time=71.3s
Epoch 7: loss=0.0023 val_auroc=0.9977 f1m=0.9816 acc=0.9816 ece=0.0133 time=70.7s
Epoch 8: loss=0.0011 val_auroc=0.9980 f1m=0.9800 acc=0.9800 ece=0.0147 time=70.5s
[INFO] Early stopping at epoch 8.
[INFO] Grad-CAM gallery placeholder for resnet50 (n=8)
[INFO] Saved classification report to /content/drive/MyDrive/biomed_pipeline_updated_run/reports/classification_report_resnet50.txt and /content/drive/MyDrive/biomed_pipeline_updated_run/reports/classification_report_resnet50.md

=== Model: convnext_tiny 

100%|██████████| 109M/109M [00:00<00:00, 260MB/s] 


Epoch 1: loss=0.2647 val_auroc=0.9951 f1m=0.9816 acc=0.9816 ece=0.0136 time=162.9s
Epoch 2: loss=0.0606 val_auroc=0.9949 f1m=0.9716 acc=0.9716 ece=0.0154 time=155.9s
Epoch 3: loss=0.0277 val_auroc=0.9915 f1m=0.9583 acc=0.9583 ece=0.0257 time=154.4s
Epoch 4: loss=0.0315 val_auroc=0.9964 f1m=0.9833 acc=0.9833 ece=0.0153 time=154.9s
Epoch 5: loss=0.0147 val_auroc=0.9952 f1m=0.9816 acc=0.9816 ece=0.0148 time=155.8s
Epoch 6: loss=0.0024 val_auroc=0.9951 f1m=0.9833 acc=0.9833 ece=0.0184 time=154.9s
Epoch 7: loss=0.0013 val_auroc=0.9952 f1m=0.9816 acc=0.9816 ece=0.0170 time=155.1s
[INFO] Early stopping at epoch 7.
[INFO] Grad-CAM gallery placeholder for convnext_tiny (n=8)
[INFO] Saved classification report to /content/drive/MyDrive/biomed_pipeline_updated_run/reports/classification_report_convnext_tiny.txt and /content/drive/MyDrive/biomed_pipeline_updated_run/reports/classification_report_convnext_tiny.md

=== Model: vgg16 ===
Downloading: "https://download.pytorch.org/models/vgg16-397923af

100%|██████████| 528M/528M [00:02<00:00, 254MB/s]


Epoch 1: loss=0.6703 val_auroc=0.8520 f1m=0.4714 acc=0.5710 ece=0.1784 time=96.2s
Epoch 2: loss=0.4921 val_auroc=0.9061 f1m=0.8178 acc=0.8180 ece=0.0434 time=97.5s
Epoch 3: loss=0.3985 val_auroc=0.9314 f1m=0.7491 acc=0.7613 ece=0.0902 time=107.4s
Epoch 4: loss=0.2937 val_auroc=0.9549 f1m=0.9031 acc=0.9032 ece=0.0265 time=107.1s
Epoch 5: loss=0.1944 val_auroc=0.9773 f1m=0.9213 acc=0.9215 ece=0.0344 time=106.6s
Epoch 6: loss=0.0969 val_auroc=0.9847 f1m=0.9365 acc=0.9366 ece=0.0425 time=107.0s
Epoch 7: loss=0.0426 val_auroc=0.9909 f1m=0.9699 acc=0.9699 ece=0.0281 time=106.7s
Epoch 8: loss=0.0197 val_auroc=0.9918 f1m=0.9699 acc=0.9699 ece=0.0240 time=105.9s
Epoch 9: loss=0.0065 val_auroc=0.9915 f1m=0.9699 acc=0.9699 ece=0.0294 time=106.3s
Epoch 10: loss=0.0070 val_auroc=0.9920 f1m=0.9749 acc=0.9750 ece=0.0268 time=96.2s
[INFO] Grad-CAM gallery placeholder for vgg16 (n=8)
[INFO] Saved classification report to /content/drive/MyDrive/biomed_pipeline_updated_run/reports/classification_report_v